In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for comprehensive data profiling of metric tables in Unity Catalog
# Purpose: Generate a metric_profile table with column-level statistics for all metric tables
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script profiles all columns in health_insurance_claims_metric, d_product_revenue_metric, and s_field_reporting_sales_source_customer_metric.
#              It computes null count, distinct count, total count, min, max, mean (where applicable), and writes results to purgo_databricks.purgo_playground.metric_profile.
#              The script includes schema validation, error handling, and test assertions for data quality and correctness.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import (  
    StringType, IntegerType, FloatType, DoubleType, ShortType, LongType, DateType, TimestampType, BooleanType
)
from pyspark.sql.utils import AnalysisException  

# ------------------ CONFIGURATION ------------------
CATALOG = "purgo_databricks"
SCHEMA = "purgo_playground"
PROFILE_TABLE = "metric_profile"
PROFILE_TABLE_PATH = f"{CATALOG}.{SCHEMA}.{PROFILE_TABLE}"

METRIC_TABLES = [
    "health_insurance_claims_metric",
    "d_product_revenue_metric",
    "s_field_reporting_sales_source_customer_metric"
]

# ------------------ UTILITY FUNCTIONS ------------------

def get_table_schema(table_full_name):
    """
    Returns the schema of the given table as a list of (column_name, data_type) tuples.
    Args:
        table_full_name (str): Fully qualified table name (e.g., 'purgo_databricks.purgo_playground.health_insurance_claims_metric')
    Returns:
        List[Tuple[str, str]]: List of (column_name, data_type) pairs
    """
    try:
        df = spark.read.table(table_full_name)
        return [(field.name, field.dataType) for field in df.schema.fields]
    except AnalysisException as e:
        raise RuntimeError(f"Table {table_full_name} not found: {str(e)}")

def get_data_type_string(dataType):
    """
    Converts a PySpark DataType object to a string representation compatible with metric_profile.
    Args:
        dataType (DataType): PySpark DataType object
    Returns:
        str: String representation of the data type
    """
    if isinstance(dataType, StringType):
        return "string"
    elif isinstance(dataType, IntegerType):
        return "int"
    elif isinstance(dataType, FloatType):
        return "float"
    elif isinstance(dataType, DoubleType):
        return "double"
    elif isinstance(dataType, ShortType):
        return "short"
    elif isinstance(dataType, LongType):
        return "bigint"
    elif isinstance(dataType, DateType):
        return "date"
    elif isinstance(dataType, TimestampType):
        return "timestamp"
    elif isinstance(dataType, BooleanType):
        return "boolean"
    else:
        raise ValueError(f"Unsupported data type: {dataType}")

def is_numeric_type(dataType):
    """
    Checks if the data type is numeric (int, float, double, short, long, bigint).
    Args:
        dataType (DataType): PySpark DataType object
    Returns:
        bool: True if numeric, False otherwise
    """
    return isinstance(dataType, (IntegerType, FloatType, DoubleType, ShortType, LongType))

def is_boolean_type(dataType):
    """
    Checks if the data type is boolean.
    Args:
        dataType (DataType): PySpark DataType object
    Returns:
        bool: True if boolean, False otherwise
    """
    return isinstance(dataType, BooleanType)

def is_date_type(dataType):
    """
    Checks if the data type is date or timestamp.
    Args:
        dataType (DataType): PySpark DataType object
    Returns:
        bool: True if date or timestamp, False otherwise
    """
    return isinstance(dataType, (DateType, TimestampType))

def is_string_type(dataType):
    """
    Checks if the data type is string.
    Args:
        dataType (DataType): PySpark DataType object
    Returns:
        bool: True if string, False otherwise
    """
    return isinstance(dataType, StringType)

def profile_column(df, column_name, dataType):
    """
    Profiles a single column and returns a dictionary of metrics.
    Args:
        df (DataFrame): Input DataFrame
        column_name (str): Name of the column to profile
        dataType (DataType): PySpark DataType object
    Returns:
        dict: Dictionary of profiling metrics
    """
    # Null count
    null_count = df.filter(F.col(column_name).isNull()).count()
    # Distinct count
    distinct_count = df.select(column_name).distinct().count()
    # Total count
    total_count = df.count()
    # Min/Max/Mean
    if is_numeric_type(dataType):
        min_value = df.select(F.min(column_name)).first()[0]
        max_value = df.select(F.max(column_name)).first()[0]
        mean = df.select(F.mean(column_name)).first()[0]
    elif is_boolean_type(dataType):
        # For boolean, min/max as 0/1, mean as average (cast to double)
        min_value = df.select(F.min(F.col(column_name).cast("int"))).first()[0]
        max_value = df.select(F.max(F.col(column_name).cast("int"))).first()[0]
        mean = df.select(F.mean(F.col(column_name).cast("double"))).first()[0]
    elif is_date_type(dataType):
        min_value = df.select(F.min(column_name)).first()[0]
        max_value = df.select(F.max(column_name)).first()[0]
        mean = None
    elif is_string_type(dataType):
        min_value = df.select(F.min(column_name)).first()[0]
        max_value = df.select(F.max(column_name)).first()[0]
        mean = None
    else:
        min_value = None
        max_value = None
        mean = None
    # Cast min/max to string for output
    min_value_str = str(min_value) if min_value is not None else None
    max_value_str = str(max_value) if max_value is not None else None
    # For mean, ensure double or None
    mean_val = float(mean) if mean is not None and (is_numeric_type(dataType) or is_boolean_type(dataType)) else None
    return {
        "column_name": column_name,
        "data_type": get_data_type_string(dataType),
        "null_count": null_count,
        "distinct_count": distinct_count,
        "total_count": total_count,
        "min_value": min_value_str,
        "max_value": max_value_str,
        "mean": mean_val
    }

def validate_profile_row(row):
    """
    Validates a profile row for required fields and data types.
    Args:
        row (dict): Profile row dictionary
    Returns:
        None; raises AssertionError if validation fails
    """
    # Required fields
    assert row["table_name"] is not None, "table_name must not be null"
    assert row["column_name"] is not None, "column_name must not be null"
    assert row["data_type"] is not None, "data_type must not be null"
    assert isinstance(row["null_count"], int), "null_count must be int"
    assert isinstance(row["distinct_count"], int), "distinct_count must be int"
    assert isinstance(row["total_count"], int), "total_count must be int"
    # Mean must be double or None
    if row["data_type"] in ["bigint", "double", "int", "float", "short", "long", "boolean"]:
        assert (row["mean"] is None or isinstance(row["mean"], float)), "mean must be double or None for numeric/boolean"
    else:
        assert row["mean"] is None, "mean must be null for non-numeric columns"
    # Min/Max must be string or None
    assert (row["min_value"] is None or isinstance(row["min_value"], str)), "min_value must be string or None"
    assert (row["max_value"] is None or isinstance(row["max_value"], str)), "max_value must be string or None"

def get_metric_profile_schema():
    """
    Returns the schema for the metric_profile table.
    Args:
        None
    Returns:
        List[Tuple[str, DataType]]: List of (column_name, DataType) pairs
    """
    return [
        ("column_name", StringType()),
        ("data_type", StringType()),
        ("distinct_count", LongType()),
        ("max_value", StringType()),
        ("mean", DoubleType()),
        ("min_value", StringType()),
        ("null_count", LongType()),
        ("table_name", StringType()),
        ("total_count", LongType())
    ]

def create_metric_profile_df(profile_rows):
    """
    Creates a DataFrame from a list of profile row dictionaries, matching the metric_profile schema.
    Args:
        profile_rows (List[dict]): List of profile row dictionaries
    Returns:
        DataFrame: PySpark DataFrame with correct schema
    """
    from pyspark.sql import Row  
    schema = get_metric_profile_schema()
    fields = [name for name, _ in schema]
    rows = []
    for r in profile_rows:
        # Ensure all fields present
        row = [r.get(f, None) for f in fields]
        rows.append(Row(*row))
    # Build DataFrame
    df = spark.createDataFrame(rows, schema=[F for F in schema])
    return df

def overwrite_delta_table(df, table_path):
    """
    Overwrites the target Delta table with the given DataFrame.
    Args:
        df (DataFrame): DataFrame to write
        table_path (str): Fully qualified table name
    Returns:
        None
    """
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_path)

def assert_table_schema_matches(df, table_path):
    """
    Asserts that the DataFrame schema matches the target table schema.
    Args:
        df (DataFrame): DataFrame to check
        table_path (str): Fully qualified table name
    Returns:
        None; raises AssertionError if mismatch
    """
    table_df = spark.read.table(table_path)
    df_fields = [f.name for f in df.schema.fields]
    table_fields = [f.name for f in table_df.schema.fields]
    assert df_fields == table_fields, f"Schema mismatch: DataFrame columns {df_fields} != Table columns {table_fields}"

# ------------------ MAIN PROFILING LOGIC ------------------

profile_rows = []
for table_name in METRIC_TABLES:
    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    try:
        df = spark.read.table(full_table_name)
    except AnalysisException as e:
        raise RuntimeError(f"Table {table_name} not found in {CATALOG}.{SCHEMA}")
    schema = df.schema
    for field in schema.fields:
        col_name = field.name
        data_type = field.dataType
        # Only allow supported types
        try:
            dtype_str = get_data_type_string(data_type)
        except ValueError as e:
            raise RuntimeError(f"Unsupported data type {data_type} for column {col_name}")
        # Profile column
        metrics = profile_column(df, col_name, data_type)
        metrics["table_name"] = table_name
        # Validate row
        validate_profile_row(metrics)
        profile_rows.append(metrics)

# ------------------ DATAFRAME CREATION AND SCHEMA VALIDATION ------------------

profile_df = create_metric_profile_df(profile_rows)
assert_table_schema_matches(profile_df, PROFILE_TABLE_PATH)

# ------------------ DELTA LAKE WRITE ------------------

overwrite_delta_table(profile_df, PROFILE_TABLE_PATH)

# ------------------ INTEGRATION TESTS ------------------

def test_metric_profile_row_counts():
    """
    Asserts that the metric_profile table has the correct number of rows (one per column per table).
    Args:
        None
    Returns:
        None; raises AssertionError if mismatch
    """
    expected_rows = sum([len(get_table_schema(f"{CATALOG}.{SCHEMA}.{t}")) for t in METRIC_TABLES])
    actual_rows = spark.read.table(PROFILE_TABLE_PATH).count()
    assert actual_rows == expected_rows, f"Row count mismatch: expected {expected_rows}, got {actual_rows}"

def test_metric_profile_no_duplicates():
    """
    Asserts that there are no duplicate profile rows for the same table_name and column_name.
    Args:
        None
    Returns:
        None; raises AssertionError if duplicates found
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    dup_count = df.groupBy("table_name", "column_name").count().filter(F.col("count") > 1).count()
    assert dup_count == 0, "Duplicate profile rows detected"

def test_metric_profile_required_fields():
    """
    Asserts that all required fields are present and not null (except mean/min/max for non-numeric types).
    Args:
        None
    Returns:
        None; raises AssertionError if missing fields
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    required = ["table_name", "column_name", "data_type", "null_count", "distinct_count", "total_count"]
    for col in required:
        nulls = df.filter(F.col(col).isNull()).count()
        assert nulls == 0, f"Required field {col} has nulls"

def test_metric_profile_mean_null_for_non_numeric():
    """
    Asserts that mean is null for non-numeric columns.
    Args:
        None
    Returns:
        None; raises AssertionError if mean is not null for non-numeric columns
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    non_numeric_types = ["string", "date", "timestamp"]
    for dtype in non_numeric_types:
        count = df.filter((F.col("data_type") == dtype) & (F.col("mean").isNotNull())).count()
        assert count == 0, f"Mean is not null for non-numeric type {dtype}"

def test_metric_profile_total_count_consistency():
    """
    Asserts that total_count matches the row count of the source table for each column.
    Args:
        None
    Returns:
        None; raises AssertionError if mismatch
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    for table_name in METRIC_TABLES:
        src_count = spark.read.table(f"{CATALOG}.{SCHEMA}.{table_name}").count()
        profile_counts = df.filter(F.col("table_name") == table_name).select("total_count").distinct().collect()
        for row in profile_counts:
            assert row["total_count"] == src_count, f"total_count mismatch for {table_name}: expected {src_count}, got {row['total_count']}"

def test_metric_profile_distinct_count():
    """
    Asserts that distinct_count matches the actual distinct count for each column.
    Args:
        None
    Returns:
        None; raises AssertionError if mismatch
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    for table_name in METRIC_TABLES:
        src_df = spark.read.table(f"{CATALOG}.{SCHEMA}.{table_name}")
        for field in src_df.schema.fields:
            col_name = field.name
            expected = src_df.select(col_name).distinct().count()
            actual = df.filter((F.col("table_name") == table_name) & (F.col("column_name") == col_name)).select("distinct_count").first()["distinct_count"]
            assert actual == expected, f"distinct_count mismatch for {table_name}.{col_name}: expected {expected}, got {actual}"

def test_metric_profile_null_count():
    """
    Asserts that null_count matches the actual null count for each column.
    Args:
        None
    Returns:
        None; raises AssertionError if mismatch
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    for table_name in METRIC_TABLES:
        src_df = spark.read.table(f"{CATALOG}.{SCHEMA}.{table_name}")
        for field in src_df.schema.fields:
            col_name = field.name
            expected = src_df.filter(F.col(col_name).isNull()).count()
            actual = df.filter((F.col("table_name") == table_name) & (F.col("column_name") == col_name)).select("null_count").first()["null_count"]
            assert actual == expected, f"null_count mismatch for {table_name}.{col_name}: expected {expected}, got {actual}"

def test_metric_profile_min_max_string():
    """
    Asserts that min_value and max_value for string columns are lexicographically smallest/largest.
    Args:
        None
    Returns:
        None; raises AssertionError if mismatch
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    for table_name in METRIC_TABLES:
        src_df = spark.read.table(f"{CATALOG}.{SCHEMA}.{table_name}")
        for field in src_df.schema.fields:
            col_name = field.name
            dtype = get_data_type_string(field.dataType)
            if dtype == "string":
                expected_min = src_df.select(F.min(col_name)).first()[0]
                expected_max = src_df.select(F.max(col_name)).first()[0]
                row = df.filter((F.col("table_name") == table_name) & (F.col("column_name") == col_name)).select("min_value", "max_value").first()
                assert row["min_value"] == str(expected_min), f"min_value mismatch for {table_name}.{col_name}: expected {expected_min}, got {row['min_value']}"
                assert row["max_value"] == str(expected_max), f"max_value mismatch for {table_name}.{col_name}: expected {expected_max}, got {row['max_value']}"

def test_metric_profile_min_max_date():
    """
    Asserts that min_value and max_value for date columns are earliest/latest date.
    Args:
        None
    Returns:
        None; raises AssertionError if mismatch
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    for table_name in METRIC_TABLES:
        src_df = spark.read.table(f"{CATALOG}.{SCHEMA}.{table_name}")
        for field in src_df.schema.fields:
            col_name = field.name
            dtype = get_data_type_string(field.dataType)
            if dtype == "date":
                expected_min = src_df.select(F.min(col_name)).first()[0]
                expected_max = src_df.select(F.max(col_name)).first()[0]
                row = df.filter((F.col("table_name") == table_name) & (F.col("column_name") == col_name)).select("min_value", "max_value").first()
                assert row["min_value"] == str(expected_min), f"min_value mismatch for {table_name}.{col_name}: expected {expected_min}, got {row['min_value']}"
                assert row["max_value"] == str(expected_max), f"max_value mismatch for {table_name}.{col_name}: expected {expected_max}, got {row['max_value']}"

def test_metric_profile_mean_numeric():
    """
    Asserts that mean for numeric columns is the average of non-null values.
    Args:
        None
    Returns:
        None; raises AssertionError if mismatch
    """
    df = spark.read.table(PROFILE_TABLE_PATH)
    for table_name in METRIC_TABLES:
        src_df = spark.read.table(f"{CATALOG}.{SCHEMA}.{table_name}")
        for field in src_df.schema.fields:
            col_name = field.name
            dtype = get_data_type_string(field.dataType)
            if dtype in ["bigint", "double", "int", "float", "short", "long"]:
                expected = src_df.select(F.mean(col_name)).first()[0]
                row = df.filter((F.col("table_name") == table_name) & (F.col("column_name") == col_name)).select("mean").first()
                if expected is not None:
                    assert abs(row["mean"] - expected) < 1e-6, f"mean mismatch for {table_name}.{col_name}: expected {expected}, got {row['mean']}"

# ------------------ RUN ALL TESTS ------------------

test_metric_profile_row_counts()
test_metric_profile_no_duplicates()
test_metric_profile_required_fields()
test_metric_profile_mean_null_for_non_numeric()
test_metric_profile_total_count_consistency()
test_metric_profile_distinct_count()
test_metric_profile_null_count()
test_metric_profile_min_max_string()
test_metric_profile_min_max_date()
test_metric_profile_mean_numeric()

# spark.stop()  # Do not stop SparkSession in Databricks
